# VoiceGuard AI — Phase 2 inspection
Run from the repository root in Colab or Jupyter after preprocessing.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT / 'voiceguard-ai').exists(): ROOT /= 'voiceguard-ai'
metadata = pd.read_csv(ROOT / 'data/metadata/processed_metadata.csv')
validation = pd.read_csv(ROOT / 'data/metadata/validation_report.csv')
metadata.head()

In [ ]:
display(metadata['label_name'].value_counts())
display(validation['duration'].describe())
validation['duration'].dropna().hist(bins=40); plt.title('Source duration (seconds)'); plt.show()
display(validation['sample_rate'].value_counts().sort_index())

In [ ]:
# Listen locally only; no audio is uploaded by this notebook.
for path in metadata['file_path'].head(3):
    display(Audio(str(Path(path) if Path(path).is_absolute() else ROOT / path)))

In [ ]:
# Verify standardization without loading the dataset into memory.
checks = []
for path in metadata['file_path'].head(100):
    info = sf.info(str(Path(path) if Path(path).is_absolute() else ROOT / path))
    checks.append((info.samplerate, info.channels))
assert all(item == (16000, 1) for item in checks)
print('Checked files are 16 kHz mono:', len(checks))

In [ ]:
# A source must never cross splits; duplicate hashes are surfaced for review.
assert metadata.groupby('source_file')['split'].nunique().max() <= 1
hash_splits = metadata.groupby('source_file_hash')['split'].nunique()
display(hash_splits[hash_splits > 1].rename('split_count').to_frame())
display(metadata.groupby(['split', 'label_name']).size().unstack(fill_value=0))
display(metadata.sample(min(10, len(metadata)), random_state=42))